In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import anndata as ad
import squidpy as sq
import cellcharter as cc
import pandas as pd
import scanpy as sc
import scvi
import numpy as np
import matplotlib.pyplot as plt
from lightning.pytorch import seed_everything
import glob
from os.path import join

seed_everything(12345)
scvi.settings.seed = 12345

In [ ]:
def run_cellcharter_atac(adata, batch_key=None, n_neighs=None, n_clusters=None):
    sc.pp.filter_genes(adata, min_cells=int(adata.n_obs * 0.01))
    adata.layers["counts"] = adata.X.copy()
    scvi.model.SCVI.setup_anndata(
        adata, 
        layer="counts", 
        batch_key=batch_key)
    model = scvi.model.SCVI(adata, gene_likelihood='poisson')
    model.train(early_stopping=True, enable_progress_bar=True, batch_size=129)
    adata.obsm['X_scVI'] = model.get_latent_representation(adata).astype(np.float32)
    if n_neighs is None:
        sq.gr.spatial_neighbors(adata, library_key='Slice', coord_type='generic', delaunay=True, spatial_key='spatial', percentile=99)
    else:
        sq.gr.spatial_neighbors(adata, library_key='Slice', coord_type='grid', n_neighs=n_neighs, spatial_key='spatial')
    cc.gr.aggregate_neighbors(adata, n_layers=3, use_rep='X_scVI', out_key='X_cellcharter', sample_key='Slice')
    gmm = cc.tl.Cluster(
        n_clusters=n_clusters, 
        random_state=12345,
        # If running on GPU
        #trainer_params=dict(accelerator='gpu', devices=1)
    )
    # Then we fit the model to the data and predict the cluster label of all cells.
    gmm.fit(adata, use_rep='X_cellcharter')
    adata.obs['cluster_cellcharter_'+str(n_clusters)] = gmm.predict(adata, use_rep='X_cellcharter')
    return adata

In [ ]:
emb_save_path = "../../../results/embeddings/"
clust_save_path = "../../../results/clusters/"
base_path = "../../../data/processed/"

In [ ]:
data_names = [
    "Misar-Stereo", "DBiT-RnaAtac-Atac", "Mux-VisiumHD-HE", "Embryo",
    "Misar-E13-E15-E18", "Misar-E15-E18-E18",
    "Lymph-S1-S2-S3", "Lymph-S1-S2-S2",
    "Tonsil-S1-S2-S3", "Tonsil-S1-S2-S2",
    "MB-5M", "MB-H3K4me3"
]

neighbors_map = { 
    "Misar-E13-E15-E18": 4, 
    "Misar-E15-E18-E18": 4, 
    "Lymph-S1-S2-S3":6, 
    "Lymph-S1-S2-S2":6, 
    "Tonsil-S1-S2-S3":6, 
    "Tonsil-S1-S2-S2":6, 
    "MB-5M":4, 
    "MB-H3K4me3":4,
    "DBiT-RnaAtac-Atac":8,
    "Misar-Stereo":8,
    "Mux-VisiumHD-HE":6,
    "Embryo":8
}

clusters_map = { 
    "Misar-E13-E15-E18": 14, 
    "Misar-E15-E18-E18": 14, 
    "Lymph-S1-S2-S3":10, 
    "Lymph-S1-S2-S2":10, 
    "Tonsil-S1-S2-S3":4, 
    "Tonsil-S1-S2-S2":4, 
    "MB-5M":15, 
    "MB-H3K4me3":13,
    "DBiT-RnaAtac-Atac":15,
    "Misar-Stereo":18,
    "Mux-VisiumHD-HE":18,
    "Embryo":13
}

prefix_all = ["atac",  "h3k27me3", "h3k27ac",  "h3k4me3"]
p2s = ['ATAC', 'H3K27me3', 'H3K27ac', 'H3K4me3']

for prefix, p2 in zip(prefix_all, p2s):
    for data_name in data_names:
        print(f"[--------]: {data_name}")
        data_path = join(base_path, data_name)
        files = glob.glob(data_path+"/*/ad_"+prefix+"*.h5ad")
        if len(files) > 0:
            adatas = []
            for file in files:
                adata = sc.read_h5ad(file)
                if "Slice" not in adata.obs.columns:
                    fallback_slice = file.split("/")[-2]
                    adata.obs["Slice"] = fallback_slice
                adatas.append(adata)
            adata = sc.concat(adatas)
            adata.obs["Slice"] = adata.obs["Slice"].astype("category")
            
            adata = run_cellcharter_atac(adata, batch_key="Slice", n_neighs=neighbors_map[data_name], n_clusters=clusters_map[data_name])
            
            os.makedirs(join(emb_save_path, f'CellCharter-{p2}', data_name), exist_ok=True)
            emb = adata.obsm["X_cellcharter"]
            emb = pd.DataFrame(emb, index=adata.obs_names, columns=[i for i in range(emb.shape[1])])
            emb.to_csv(join(emb_save_path, f'CellCharter-{p2}', data_name, 'df_emb.csv'))
            
            os.makedirs(join(clust_save_path, f'CellCharter-{p2}', data_name), exist_ok=True)
            label = adata.obs[['cluster_cellcharter_'+str(clusters_map[data_name])]].copy()
            label.columns = ["cluster"]
            label.to_csv(join(clust_save_path, f'CellCharter-{p2}', data_name, 'df_cluster.csv'))